# Knowledge Graph Results Viewer

Read-only notebook for inspecting knowledge-graph outputs after a pipeline run.

It loads tables from a project directory like `outputs/kg_cia_smoke_para50/` and previews:

- `extraction/`
- `graphs/`
- `hierarchy/`
- optional inline HTML visualizations from `hierarchy/visualizations/`


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 20)


def find_repo_root(start=None):
    current = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "knowledge_graph" / "run_pipeline.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing knowledge_graph/run_pipeline.py")


def load_csv(path):
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f"Loaded {path} -> {df.shape}")
    return df


def preview(df, preferred_cols=None, n=10):
    if df.empty:
        print("Empty table")
        return
    preferred_cols = preferred_cols or []
    cols = [col for col in preferred_cols if col in df.columns]
    display(df[cols].head(n) if cols else df.head(n))


def rel(path, repo_root):
    try:
        return path.resolve().relative_to(repo_root.resolve()).as_posix()
    except Exception:
        return str(path)


In [2]:
REPO_ROOT = find_repo_root()

# Change this to the project directory you want to inspect.
# PROJECT_DIR = REPO_ROOT / "outputs" / "kg_cia_smoke_para50"
PROJECT_DIR = REPO_ROOT / "outputs" / "kg_cia_smoke_doc50"

# Set True if you want to inline the generated PyVis HTML inside the notebook.
SHOW_HTML_VISUALIZATIONS = False

print("REPO_ROOT:", REPO_ROOT)
print("PROJECT_DIR:", PROJECT_DIR)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory does not exist: {PROJECT_DIR}")


REPO_ROOT: /Users/jimyhc/Desktop/research/IARPA
PROJECT_DIR: /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50


In [3]:
table_paths = {
    "entities": PROJECT_DIR / "extraction" / "entities.csv",
    "events": PROJECT_DIR / "extraction" / "events.csv",
    "claims": PROJECT_DIR / "extraction" / "claims.csv",
    "relations": PROJECT_DIR / "extraction" / "relations.csv",
    "canonical_nodes": PROJECT_DIR / "graphs" / "canonical_nodes_cluster.csv",
    "analysis_edges": PROJECT_DIR / "graphs" / "analysis_edges_cluster.csv",
    "node_to_level1": PROJECT_DIR / "graphs" / "node_to_level1.csv",
    "level1_summary": PROJECT_DIR / "graphs" / "level1_summary.csv",
    "node_to_level2": PROJECT_DIR / "graphs" / "node_to_level2.csv",
    "level2_summary": PROJECT_DIR / "graphs" / "level2_summary.csv",
    "community_nodes": PROJECT_DIR / "hierarchy" / "community_nodes.csv",
    "community_edges": PROJECT_DIR / "hierarchy" / "community_edges.csv",
    "community_to_narrative": PROJECT_DIR / "hierarchy" / "community_to_narrative.csv",
    "narrative_nodes": PROJECT_DIR / "hierarchy" / "narrative_nodes.csv",
    "narrative_edges": PROJECT_DIR / "hierarchy" / "narrative_edges.csv",
    "hierarchy_index": PROJECT_DIR / "hierarchy" / "hierarchy_index.csv",
}

tables = {name: load_csv(path) for name, path in table_paths.items()}


Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50/extraction/entities.csv -> (645, 12)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50/extraction/events.csv -> (254, 13)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50/extraction/claims.csv -> (177, 13)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50/extraction/relations.csv -> (228, 11)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50/graphs/canonical_nodes_cluster.csv -> (542, 8)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50/graphs/analysis_edges_cluster.csv -> (4004, 9)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50/graphs/node_to_level1.csv -> (542, 2)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50/graphs/level1_summary.csv -> (11, 5)
Loaded /Users/jimyhc/Desktop/research/IARPA/outputs/kg_cia_smoke_doc50/graphs/node_to_level2.csv -> (11, 2)
Loaded /Us

In [4]:
summary_rows = []
for name, df in tables.items():
    summary_rows.append({"table": name, "rows": len(df), "columns": len(df.columns)})

summary_df = pd.DataFrame(summary_rows).sort_values(["rows", "table"], ascending=[False, True]).reset_index(drop=True)
display(summary_df)

high_level = {
    "unique_entity_labels": tables["entities"]["label"].nunique() if "label" in tables["entities"].columns else 0,
    "unique_event_labels": tables["events"]["label"].nunique() if "label" in tables["events"].columns else 0,
    "unique_docs_in_entities": tables["entities"]["doc_id"].nunique() if "doc_id" in tables["entities"].columns else 0,
    "communities": len(tables["community_nodes"]),
    "narratives": len(tables["narrative_nodes"]),
}
display(pd.DataFrame([high_level]))


,table,rows,columns
0,analysis_edges,4004,9
1,entities,645,12
2,canonical_nodes,542,8
3,node_to_level1,542,2
4,events,254,13
5,relations,228,11
6,claims,177,13
7,community_edges,32,8
8,community_nodes,11,8
9,community_to_narrative,11,4


,unique_entity_labels,unique_event_labels,unique_docs_in_entities,communities,narratives
0,296,250,50,11,1


## Extraction Preview

In [5]:
preview(tables["entities"], ["doc_id", "paragraph_id", "label", "type", "description", "confidence"], n=15)
preview(tables["events"], ["doc_id", "paragraph_id", "label", "type", "date", "location", "confidence"], n=15)
preview(tables["claims"], ["doc_id", "paragraph_id", "claim_text", "speaker", "target", "stance", "confidence"], n=15)
preview(tables["relations"], ["doc_id", "paragraph_id", "source", "target", "relation", "confidence"], n=15)


,doc_id,paragraph_id,label,type,description,confidence
0,2000168_CIA-RDP79T00975A005500040001-6,doc0,Turkey,Country,"Country in Asia and Europe, involved in political changes.",0.7
1,2000168_CIA-RDP79T00975A005500040001-6,doc0,President Gursel,Person,"President of Turkey, involved in cabinet changes.",0.7
2,2000168_CIA-RDP79T00975A005500040001-6,doc0,UAR (Syria),Government,"United Arab Republic, involved in political unrest.",0.7
3,2000168_CIA-RDP79T00975A005500040001-6,doc0,Jordan,Country,Country providing support to Syrian dissidents.,0.7
4,2000168_CIA-RDP79T00975A005500040001-6,doc0,Pakistan,Country,Country reassessing its foreign relations.,0.7
5,2000168_CIA-RDP79T00975A005500040001-6,doc0,General Ne Win,Person,Burmese General under pressure to take control.,0.7
6,2000168_CIA-RDP79T00975A005500040001-6,doc0,Burma,Country,Country experiencing political instability.,0.7
7,2000168_CIA-RDP79T00975A005500040001-6,doc0,Republic of the Congo,Country,Country involved in military and political issues.,0.7
8,2000168_CIA-RDP79T00975A005500040001-6,doc0,Ecuador,Country,Country engaging in diplomatic talks with the Soviet Union.,0.7
9,2000168_CIA-RDP79T00975A005500040001-6,doc0,Jose Chiriboga,Person,Ecuadorean Foreign Minister involved in discussions with the Soviet ambassador.,0.7


,doc_id,paragraph_id,label,type,date,location,confidence
0,2000168_CIA-RDP79T00975A005500040001-6,doc0,Cabinet Resignation in Turkey,PolicyAction,1961-01-04,Turkey,0.7
1,2000168_CIA-RDP79T00975A005500040001-6,doc0,Support for Syrian Dissidents,Communication,NaN,Jordan,0.7
2,2000168_CIA-RDP79T00975A005500040001-6,doc0,Airlift Flights into Laos,MilitaryAction,1961-01-04,Laos,0.7
3,2000168_CIA-RDP79T00975A005500040001-6,doc0,Meeting of Constituent Assembly,Meeting,1961-01-06,Turkey,0.7
4,2000168_CIA-RDP79T00975A005500040001-6,doc0,Ecuadorean Foreign Minister's Meeting,Meeting,1961-01-04,Washington,0.7
5,2000169_CIA-RDP79T00975A005500060001-4,doc0,Severance of diplomatic relations,PolicyAction,1961-01-05,Nigeria,0.7
6,2000169_CIA-RDP79T00975A005500060001-4,doc0,Postponement of Soviet party congress,PolicyAction,1961-01-02,Moscow,0.7
7,2000169_CIA-RDP79T00975A005500060001-4,doc0,Military training mission proposal,Negotiation,NaN,Somali Republic,0.7
8,2000169_CIA-RDP79T00975A005500060001-4,doc0,Diplomatic relations request,Communication,1961-01-11,Nigeria,0.7
9,2000169_CIA-RDP79T00975A005500060001-4,doc0,Diplomatic relations request,Communication,NaN,Dominican Republic,0.7


,doc_id,paragraph_id,claim_text,speaker,target,stance,confidence
0,2000168_CIA-RDP79T00975A005500040001-6,doc0,The Boun Oum government will probably be viewed by the Communist bloc as illegal.,Central Intelligence Bulletin,Boun Oum government,neutral,0.7
1,2000168_CIA-RDP79T00975A005500040001-6,doc0,Pakistan's military government probably approved editorials calling for disengagement from Western alliances.,Central Intelligence Bulletin,Pakistan's military government,neutral,0.7
2,2000168_CIA-RDP79T00975A005500040001-6,doc0,General Ne Win may resume control of the government due to pressure from army leaders.,Central Intelligence Bulletin,General Ne Win,neutral,0.7
3,2000169_CIA-RDP79T00975A005500060001-4,doc0,Khrushchev cited need to establish contact with new US administration as reason for postponing Soviet party congress.,Khrushchev,US administration,neutral,0.7
4,2000169_CIA-RDP79T00975A005500060001-4,doc0,Nigerian attitudes toward the West were affected by Western support for Kasavubu over Lumumba.,Nigerian officials,Western nations,oppose,0.7
5,2000169_CIA-RDP79T00975A005500060001-4,doc0,Trujillo's regime is advised to align with the Soviet bloc to withstand US pressure.,Trujillo's aides,Soviet bloc,support,0.7
6,2000170_CIA-RDP79T00975A005500070001-3,doc0,The USSR is transporting military personnel into Laos.,US allegations,USSR,oppose,0.7
7,2000170_CIA-RDP79T00975A005500070001-3,doc0,The Boun Oum regime was imposed by force.,Souvanna Phouma,Boun Oum government,support,0.7
8,2000170_CIA-RDP79T00975A005500070001-3,doc0,Financial aid to Jordan would arouse considerable opposition among Iraqis.,diplomatic circles in Baghdad,Iraqi public,oppose,0.7
9,2000171_CIA-RDP79T00975A005500080001-2,doc0,Food shortages in Communist China are causing popular discontent.,Central Intelligence Bulletin,Communist China,support,0.7


,doc_id,paragraph_id,source,target,relation,confidence
0,2000168_CIA-RDP79T00975A005500040001-6,doc0,President Gursel,Turkey,AFFILIATION,0.7
1,2000168_CIA-RDP79T00975A005500040001-6,doc0,General Ne Win,Burma,EVENT_PARTICIPANT,0.7
2,2000168_CIA-RDP79T00975A005500040001-6,doc0,Jose Chiriboga,Ecuador,AFFILIATION,0.7
3,2000169_CIA-RDP79T00975A005500060001-4,doc0,Khrushchev,US administration,REQUEST,0.7
4,2000169_CIA-RDP79T00975A005500060001-4,doc0,Nigeria,France,OPPOSITION,0.7
5,2000169_CIA-RDP79T00975A005500060001-4,doc0,Somali Republic,UAR,SUPPORT,0.7
6,2000169_CIA-RDP79T00975A005500060001-4,doc0,Dominican Republic,USSR,REQUEST,0.7
7,2000169_CIA-RDP79T00975A005500060001-4,doc0,Balewa,Soviet ambassador,REQUEST,0.7
8,2000170_CIA-RDP79T00975A005500070001-3,doc0,Jordan,Iraq,REQUEST,0.7
9,2000170_CIA-RDP79T00975A005500070001-3,doc0,Belgium,Socialist-led strikes,EVENT_PARTICIPANT,0.7


## Graph Preview

In [6]:
preview(tables["canonical_nodes"], ["node_key", "label", "node_type", "mention_count", "source_type"], n=20)
preview(tables["analysis_edges"], ["source", "target", "rel", "weight", "support_count"], n=20)
preview(tables["level1_summary"], ["level1_id", "display_label", "size", "top_labels", "top_types"], n=20)
preview(tables["level2_summary"], ["level2_id", "display_label", "size", "top_labels", "top_types"], n=20)


,node_key,label,node_type,mention_count,source_type
0,CANONICAL_ENTITY:ABDULLAH_RIMAWI,ABDULLAH RIMAWI,CanonicalEntity,1.0,Person
1,CANONICAL_ENTITY:ADENAUER,ADENAUER,CanonicalEntity,2.0,Person
2,CANONICAL_ENTITY:AFGHANISTAN,AFGHANISTAN,CanonicalEntity,2.0,Country
3,CANONICAL_ENTITY:AIDIT,AIDIT,CanonicalEntity,1.0,Person
4,CANONICAL_ENTITY:AIR_GUINEA,AIR GUINEA,CanonicalEntity,1.0,Organization
5,CANONICAL_ENTITY:AKEL,AKEL,CanonicalEntity,1.0,PoliticalGroup
6,CANONICAL_ENTITY:ALBANIA,ALBANIA,CanonicalEntity,4.0,Country
7,CANONICAL_ENTITY:ALBERTO_LLERAS,ALBERTO LLERAS,CanonicalEntity,1.0,Person
8,CANONICAL_ENTITY:ALFONS_GORBACH,ALFONS GORBACH,CanonicalEntity,1.0,Person
9,CANONICAL_ENTITY:ALGERIA,ALGERIA,CanonicalEntity,3.0,Country; Location


,rel,weight,support_count
0,CO_OCCURRENCE,0.075,1
1,CO_OCCURRENCE,0.075,1
2,CO_OCCURRENCE,0.075,1
3,CO_OCCURRENCE,0.075,1
4,CO_OCCURRENCE,0.075,1
5,CO_OCCURRENCE,0.075,1
6,CO_OCCURRENCE,0.075,1
7,SHARED_ENTITY_EVENT_LINK,0.245,1
8,SHARED_ENTITY_EVENT_LINK,0.245,1
9,SHARED_ENTITY_EVENT_LINK,0.245,1


,level1_id,display_label,size,top_labels,top_types
0,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA,99,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; ARAB LEAGUE; AVERKY ARISTOV; BEN-GURION; BOLIVIA; CABINDA; COLOMBIA; CONG...,"{""EventCluster"": 54, ""CanonicalEntity"": 45}"
1,1,AFGHANISTAN; ALGERIAN REBELS; ARGENTINA; BELGIAN DEFENSE MINISTRY,80,AFGHANISTAN; ALGERIAN REBELS; ARGENTINA; BELGIAN DEFENSE MINISTRY; BELGIUM; BRAZILIAN GOVERNMENT; CHINESE NATIONALIST IRREGULARS; COLONE...,"{""CanonicalEntity"": 48, ""EventCluster"": 32}"
2,2,AIDIT; AIR GUINEA; BREZHNEV; CZECHOSLOVAKIA,61,AIDIT; AIR GUINEA; BREZHNEV; CZECHOSLOVAKIA; DAI VIET PARTY; DIEM; DUVALIER; FEDERATION OF RHODESIA AND NYASALAND; GIZENGA; GROMYKO; GUI...,"{""CanonicalEntity"": 31, ""EventCluster"": 30}"
3,3,ANICET KASHAMURA; BALEWA; BELGIAN GOVERNMENT; CHANG MYON,52,ANICET KASHAMURA; BALEWA; BELGIAN GOVERNMENT; CHANG MYON; CHIANG KAI-SHEK; COMIBOL; DEFENSE MINISTER BOTELHO MONIZ; DOMINICAN REPUBLIC; ...,"{""EventCluster"": 27, ""CanonicalEntity"": 25}"
4,4,BOUN OUM; BRAZIL; CAIRO; CASTILLO NAVARRETE,52,BOUN OUM; BRAZIL; CAIRO; CASTILLO NAVARRETE; CHEN YI; CHILE; COMMON MARKET; CUBA; EVIAN; FERHAT ABBAS; FIDEL CASTRO; ITALY; KATANGA; KHA...,"{""CanonicalEntity"": 32, ""EventCluster"": 20}"
5,5,ADENAUER; BENEDIKTOV; BOMBOKO; BURMA,51,ADENAUER; BENEDIKTOV; BOMBOKO; BURMA; CENTRAL INTELLIGENCE AGENCY; CEYLON; DUTT; GENERAL NE WIN; GURSEL; G. P. MALALASEKERA; HAMMARSKJOL...,"{""CanonicalEntity"": 27, ""EventCluster"": 24}"
6,6,ALFONS GORBACH; ALGERIA; ALGERIAN NATIONAL LIBERATION FRONT (FLN); ANSAR,41,ALFONS GORBACH; ALGERIA; ALGERIAN NATIONAL LIBERATION FRONT (FLN); ANSAR; BEIJING; BOURGUIBA; BRITISH CAMEROONS; DE GAULLE; GENERAL NASU...,"{""CanonicalEntity"": 21, ""EventCluster"": 20}"
7,7,BADR; BASRA; BOMBAY; BOUN OUM GOVERNMENT,36,BADR; BASRA; BOMBAY; BOUN OUM GOVERNMENT; DEFENSE MINISTER AKIF FAYIZ; EYSKENS; FRAIN; IMAM; IRAQ; JASON SENDWE; JORDAN; JULIUS NYERERE;...,"{""CanonicalEntity"": 24, ""EventCluster"": 12}"
8,8,ABDULLAH RIMAWI; ALBANIA; ALI ABU NUWAR; CZECHOSLOVAK AIRLINES,34,ABDULLAH RIMAWI; ALBANIA; ALI ABU NUWAR; CZECHOSLOVAK AIRLINES; GREECE; INTERNATIONAL CONTROL COMMISSION; IRAN; JORDANIAN MONARCHY; MORO...,"{""CanonicalEntity"": 18, ""EventCluster"": 16}"
9,9,AYUB KHAN; CHIRIBOGA; ECUADOR; GUATEMALA,21,AYUB KHAN; CHIRIBOGA; ECUADOR; GUATEMALA; INDONESIAN PRESIDENT SUKARNO; LEONID SEDOV; LIBERAL-DEMOCRATIC PARTY (LDP); MARIO MENDEZ MONTE...,"{""CanonicalEntity"": 13, ""EventCluster"": 8}"


,level2_id,display_label,size,top_labels,top_types
0,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN; ALGERIAN REBELS; ARGENTINA; BELGIAN DEFENSE MINISTRY; AIDIT;...,11,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN; ALGERIAN REBELS; ARGENTINA; BELGIAN DEFENSE MINISTRY; AIDIT;...,"{""LEVEL1_COMMUNITY"": 11}"


## Hierarchy Preview

In [7]:
preview(tables["community_nodes"], ["community_id", "label", "analyst_label", "size", "top_labels", "analyst_summary"], n=20)
preview(tables["narrative_nodes"], ["narrative_id", "label", "analyst_label", "num_communities", "top_labels", "analyst_summary"], n=20)
preview(tables["hierarchy_index"], ["community_id", "community_label", "narrative_id", "narrative_label", "community_size"], n=30)


,community_id,label,analyst_label,size,top_labels,analyst_summary
0,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA,Cold War crisis monitoring and escalation,99,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; ARAB LEAGUE; AVERKY ARISTOV; BEN-GURION; BOLIVIA; CABINDA; COLOMBIA; CONG...,This group connects 99 lower-level nodes around: ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; ARAB LEAGUE; AVERKY ARIS...
1,1,AFGHANISTAN; ALGERIAN REBELS; ARGENTINA; BELGIAN DEFENSE MINISTRY,"Military posture, security forces, and regime stability",80,AFGHANISTAN; ALGERIAN REBELS; ARGENTINA; BELGIAN DEFENSE MINISTRY; BELGIUM; BRAZILIAN GOVERNMENT; CHINESE NATIONALIST IRREGULARS; COLONE...,This group connects 80 lower-level nodes around: AFGHANISTAN; ALGERIAN REBELS; ARGENTINA; BELGIAN DEFENSE MINISTRY; BELGIUM; BRAZILIAN G...
2,2,AIDIT; AIR GUINEA; BREZHNEV; CZECHOSLOVAKIA,AIDIT; AIR GUINEA; BREZHNEV; CZECHOSLOVAKIA,61,AIDIT; AIR GUINEA; BREZHNEV; CZECHOSLOVAKIA; DAI VIET PARTY; DIEM; DUVALIER; FEDERATION OF RHODESIA AND NYASALAND; GIZENGA; GROMYKO; GUI...,This group connects 61 lower-level nodes around: AIDIT; AIR GUINEA; BREZHNEV; CZECHOSLOVAKIA; DAI VIET PARTY; DIEM; DUVALIER; FEDERATION...
3,3,ANICET KASHAMURA; BALEWA; BELGIAN GOVERNMENT; CHANG MYON,Chinese policy and regional positioning,52,ANICET KASHAMURA; BALEWA; BELGIAN GOVERNMENT; CHANG MYON; CHIANG KAI-SHEK; COMIBOL; DEFENSE MINISTER BOTELHO MONIZ; DOMINICAN REPUBLIC; ...,This group connects 52 lower-level nodes around: ANICET KASHAMURA; BALEWA; BELGIAN GOVERNMENT; CHANG MYON; CHIANG KAI-SHEK; COMIBOL; DEF...
4,4,BOUN OUM; BRAZIL; CAIRO; CASTILLO NAVARRETE,Cold War crisis monitoring and escalation,52,BOUN OUM; BRAZIL; CAIRO; CASTILLO NAVARRETE; CHEN YI; CHILE; COMMON MARKET; CUBA; EVIAN; FERHAT ABBAS; FIDEL CASTRO; ITALY; KATANGA; KHA...,This group connects 52 lower-level nodes around: BOUN OUM; BRAZIL; CAIRO; CASTILLO NAVARRETE; CHEN YI; CHILE; COMMON MARKET; CUBA.
5,5,ADENAUER; BENEDIKTOV; BOMBOKO; BURMA,ADENAUER; BENEDIKTOV; BOMBOKO; BURMA,51,ADENAUER; BENEDIKTOV; BOMBOKO; BURMA; CENTRAL INTELLIGENCE AGENCY; CEYLON; DUTT; GENERAL NE WIN; GURSEL; G. P. MALALASEKERA; HAMMARSKJOL...,This group connects 51 lower-level nodes around: ADENAUER; BENEDIKTOV; BOMBOKO; BURMA; CENTRAL INTELLIGENCE AGENCY; CEYLON; DUTT; GENERA...
6,6,ALFONS GORBACH; ALGERIA; ALGERIAN NATIONAL LIBERATION FRONT (FLN); ANSAR,ALFONS GORBACH; ALGERIA; ALGERIAN NATIONAL LIBERATION FRONT (FLN); ANSAR,41,ALFONS GORBACH; ALGERIA; ALGERIAN NATIONAL LIBERATION FRONT (FLN); ANSAR; BEIJING; BOURGUIBA; BRITISH CAMEROONS; DE GAULLE; GENERAL NASU...,This group connects 41 lower-level nodes around: ALFONS GORBACH; ALGERIA; ALGERIAN NATIONAL LIBERATION FRONT (FLN); ANSAR; BEIJING; BOUR...
7,7,BADR; BASRA; BOMBAY; BOUN OUM GOVERNMENT,"Military posture, security forces, and regime stability",36,BADR; BASRA; BOMBAY; BOUN OUM GOVERNMENT; DEFENSE MINISTER AKIF FAYIZ; EYSKENS; FRAIN; IMAM; IRAQ; JASON SENDWE; JORDAN; JULIUS NYERERE;...,This group connects 36 lower-level nodes around: BADR; BASRA; BOMBAY; BOUN OUM GOVERNMENT; DEFENSE MINISTER AKIF FAYIZ; EYSKENS; FRAIN; ...
8,8,ABDULLAH RIMAWI; ALBANIA; ALI ABU NUWAR; CZECHOSLOVAK AIRLINES,"Government leadership, cabinet politics, and state decision-making",34,ABDULLAH RIMAWI; ALBANIA; ALI ABU NUWAR; CZECHOSLOVAK AIRLINES; GREECE; INTERNATIONAL CONTROL COMMISSION; IRAN; JORDANIAN MONARCHY; MORO...,This group connects 34 lower-level nodes around: ABDULLAH RIMAWI; ALBANIA; ALI ABU NUWAR; CZECHOSLOVAK AIRLINES; GREECE; INTERNATIONAL C...
9,9,AYUB KHAN; CHIRIBOGA; ECUADOR; GUATEMALA,"Soviet policy, leadership, and bloc maneuvering",21,AYUB KHAN; CHIRIBOGA; ECUADOR; GUATEMALA; INDONESIAN PRESIDENT SUKARNO; LEONID SEDOV; LIBERAL-DEMOCRATIC PARTY (LDP); MARIO MENDEZ MONTE...,This group connects 21 lower-level nodes around: AYUB KHAN; CHIRIBOGA; ECUADOR; GUATEMALA; INDONESIAN PRESIDENT SUKARNO; LEONID SEDOV; L...


,narrative_id,label,analyst_label,num_communities,top_labels,analyst_summary
0,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,"Military posture, security forces, and regime stability",11,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN; ALGERIAN REBELS; ARGENTINA; BELGIAN DEFENSE MINISTRY; AIDIT;...,This group connects 11 communities around: ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN; ALGERIAN REBELS; ...


,community_id,community_label,narrative_id,narrative_label,community_size
0,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,99
1,1,AFGHANISTAN; ALGERIAN REBELS; ARGENTINA; BELGIAN DEFENSE MINISTRY,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,80
2,2,AIDIT; AIR GUINEA; BREZHNEV; CZECHOSLOVAKIA,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,61
3,3,ANICET KASHAMURA; BALEWA; BELGIAN GOVERNMENT; CHANG MYON,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,52
4,4,BOUN OUM; BRAZIL; CAIRO; CASTILLO NAVARRETE,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,52
5,5,ADENAUER; BENEDIKTOV; BOMBOKO; BURMA,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,51
6,6,ALFONS GORBACH; ALGERIA; ALGERIAN NATIONAL LIBERATION FRONT (FLN); ANSAR,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,41
7,7,BADR; BASRA; BOMBAY; BOUN OUM GOVERNMENT,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,36
8,8,ABDULLAH RIMAWI; ALBANIA; ALI ABU NUWAR; CZECHOSLOVAK AIRLINES,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,34
9,9,AYUB KHAN; CHIRIBOGA; ECUADOR; GUATEMALA,0,ALBERTO LLERAS; AMBASSADOR BROWN; AMBASSADOR THOMPSON; ANGOLA; AFGHANISTAN,21


## Visualization Files

In [8]:
vis_paths = [
    PROJECT_DIR / "hierarchy" / "visualizations" / "community_graph_pyvis.html",
    PROJECT_DIR / "hierarchy" / "visualizations" / "narrative_graph_pyvis.html",
]

for path in vis_paths:
    exists = path.exists()
    display(Markdown(f"- `{rel(path, REPO_ROOT)}`: {'present' if exists else 'missing'}"))

if SHOW_HTML_VISUALIZATIONS:
    for path in vis_paths:
        if path.exists():
            display(Markdown(f"### {path.name}"))
            display(HTML(path.read_text(encoding="utf-8")))


- `outputs/kg_cia_smoke_doc50/hierarchy/visualizations/community_graph_pyvis.html`: present

- `outputs/kg_cia_smoke_doc50/hierarchy/visualizations/narrative_graph_pyvis.html`: present